# Online Cinema A/B Test Analysis
This project evaluates the effectiveness of a new recommendation system using A/B testing and user engagement metrics.

## Project Overview

The purpose of this project is to evaluate the results of an A/B experiment conducted by an online cinema.

The analysis includes:

- data quality assessment;
- data cleaning;
- feature engineering;
- statistical hypothesis testing;
- product metrics analysis (DAU and WAU);
- business conclusions.ess conclusions.

## 1. Imports

Import the libraries required for data processing, statistical analysis and visualizatio.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import shapiro, mannwhitneyu
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Data

Load the dataset containing user interaction events collected during the experimen.


In [ ]:
df = pd.read_excel("data/Event_click.xlsx")
df.head()

## 3. Exploratory Data Analysis (EDA)

Before starting the analysis, inspect the dataset structure, data types and missing values.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
print(f'Duplicate rows: {df.duplicated().sum()}')

### Initial Observations

The initial inspection shows that:

- the dataset contains missing values;
- duplicate rows are absent;
- the dataset requires cleaning before further analysis.

The next step is to remove incomplete observations and prepare the data for the A/B test.

## 4. Data Cleaning

To ensure reliable experiment results, the dataset is cleaned before the statistical analysis.

The following steps are performed:

- remove missing values;
- identify users assigned to multiple experiment groups;
- exclude such users from the analysis.ups.t.

In [ ]:
# Remove missing values
df = df.dropna().copy()

print('Dataset shape after removing missing values:')
print(df.shape)

### Missing Values

Rows containing missing values were removed because incomplete observations may negatively affect the reliability of the experiment.

In [ ]:
# Number of unique groups for each user

group_count = (
    df.groupby('id_client')['flag_group']
      .nunique()
)

In [ ]:
# Users assigned to multiple experiment groups

duplicated_users = group_count[group_count > 1].index

print(f'Users assigned to multiple groups: {len(duplicated_users)}')

In [ ]:
# Remove users assigned to multiple groups

df = df[
    ~df['id_client'].isin(duplicated_users)
].copy()

print(df.shape)

### Data Cleaning Summary

The dataset was cleaned by:

- removing missing observations;
- excluding users assigned to multiple experiment groups.

The cleaned dataset is used for all subsequent analyses.

## 5. A/B Test Analysis

The objective of this analysis is to evaluate whether the new recommendation system increased the proportion of users who started watching a movie (performed at least one **play** action).

The analysis consists of the following steps:

- create a binary indicator of the play event;
- calculate conversion rates for the control and test groups;
- check the normality of the data distribution;
- choose an appropriate statistical test;
- evaluate statistical significance and interpret the results.).d.

In [ ]:
# Create binary indicator for play event

df['flag_play'] = (df['event'] == 'play').astype(int)

df.head()

In [ ]:
# Aggregate by user

user_play = (
    df.groupby(['id_client', 'flag_group'])['flag_play']
      .max()
      .reset_index(name='max_flag')
)

user_play.head()

### User-level aggregation

Several events may belong to the same user.

To avoid counting one participant multiple times, the data are aggregated at the user level.

If a user performed at least one **play** event, the value of **max_flag** equals 1; otherwise it equals 0.

In [ ]:
# Conversion rate

conversion = (
    user_play
    .groupby('flag_group')['max_flag']
    .mean()
)

print(conversion)

### Conversion rates

The conversion rate represents the proportion of users who started watching a movie in each experimental group.

In [ ]:
control = user_play[user_play['flag_group'] == 0]['max_flag']
test = user_play[user_play['flag_group'] == 1]['max_flag']

In [ ]:
print("Control group:")
print(shapiro(control))

print()

print("Test group:")
print(shapiro(test))

### Normality Assessment

The Shapiro–Wilk test indicated that the data in both groups are not normally distributed (p < 0.05). Therefore, a non-parametric statistical test was selected for further analysis.est.est.

In [ ]:
stat, p_value = mannwhitneyu(test, control)

print(f'U-statistic: {stat:.0f}')
print(f'p-value: {p_value:.4f}')

### Interpretation

The Mann–Whitney U test showed a statistically significant difference between the control and test groups (p = 0.0079). Thus, the null hypothesis is rejected.

### Conclusion

The A/B test results indicate that the new recommendation system significantly affected user conversion to movie playback. Since the difference between the groups is statistically significant (p < 0.05), the new recommendation algorithm can be considered more effective than the original version.

# 6. User Activity Analysis

This section evaluates user engagement using two common metrics:

- **DAU (Daily Active Users)** — the number of unique active users per day;
- **WAU (Weekly Active Users)** — the number of unique active users per week.

A user is considered active if they performed at least one event.

In [ ]:
# Daily Active Users (DAU)

dau = (
    df.groupby("dtime_event")["id_client"]
      .nunique()
      .reset_index(name="DAU")
)

dau.head()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(dau["dtime_event"], dau["DAU"], marker="o")
plt.title("Daily Active Users")
plt.xlabel("Date")
plt.ylabel("Users")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

### Daily Active Users

Daily user activity remains relatively stable throughout the observation period without pronounced upward or downward trends.

In [ ]:
# Week number according to ISO calendar

df["week"] = df["dtime_event"].dt.isocalendar().week

In [ ]:
wau = (
    df.groupby("week")["id_client"]
      .nunique()
      .reset_index(name="WAU")
)

wau

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(wau["week"], wau["WAU"], marker="o")
plt.title("Weekly Active Users")
plt.xlabel("ISO Week")
plt.ylabel("Users")
plt.grid(True)
plt.show()

### Weekly Active Users

Weekly activity is relatively stable across complete weeks. The decrease observed in the final week is explained by incomplete data for that period.

In [ ]:
# Merge daily and weekly metrics for visualization

dau["week"] = dau["dtime_event"].dt.isocalendar().week

activity = dau.merge(wau, on="week")

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(activity["dtime_event"], activity["DAU"], label="DAU")
plt.plot(activity["dtime_event"], activity["WAU"], label="WAU")

plt.title("DAU and WAU")
plt.xlabel("Date")
plt.ylabel("Users")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)

plt.show()

### DAU and WAU Comparison

Weekly activity is naturally higher than daily activity because it aggregates users over an entire week. Both metrics remain stable during the experiment.

# Final Conclusion

The dataset was successfully cleaned and prepared for analysis.

The A/B test revealed a statistically significant improvement in conversion for the new recommendation system. Since the data were not normally distributed, the Mann–Whitney U test was applied.

The DAU and WAU metrics indicate that user activity remained generally stable throughout the experiment. The decrease observed in the final week is likely explained by the incomplete observation period rather than reduced engagement.

Overall, the results suggest that the new recommendation algorithm performs better than the original version.gorithm.